<a href="https://colab.research.google.com/github/aarush2557/Zidio-Development/blob/master/Week____2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
data = pd.read_csv("retail_store_inventory_cleaned.csv")
data["Date"] = pd.to_datetime(data["Date"])

In [9]:
print("Number of rows and columns:", data.shape)
print()
print("First 5 rows:")
print(data.head())

Number of rows and columns: (73100, 24)

First 5 rows:
        Date Store ID Product ID     Category Region  Inventory Level  \
0 2022-01-01     S001      P0001    Groceries  North              231   
1 2022-01-01     S001      P0002         Toys  South              204   
2 2022-01-01     S001      P0003         Toys   West              102   
3 2022-01-01     S001      P0004         Toys  North              469   
4 2022-01-01     S001      P0005  Electronics   East              166   

   Units Sold  Units Ordered  Demand Forecast  Price  ...  Seasonality  Year  \
0         127             55           135.47  33.50  ...       Autumn  2022   
1         150             66           144.04  63.01  ...       Autumn  2022   
2          65             51            74.02  27.99  ...       Summer  2022   
3          61            164            62.18  32.72  ...       Autumn  2022   
4          14            135             9.26  73.64  ...       Summer  2022   

   Month  Day   Weekday  

In [10]:
print()
print("Units Sold - basic stats")
print(data["Units Sold"].describe())



Units Sold - basic stats
count    73100.000000
mean       136.464870
std        108.919406
min          0.000000
25%         49.000000
50%        107.000000
75%        203.000000
max        499.000000
Name: Units Sold, dtype: float64


In [11]:

plt.figure(figsize=(8,5))
plt.hist(data["Units Sold"], bins=40, color="steelblue", edgecolor="black")
plt.title("Distribution of Daily Units Sold (all rows)")
plt.xlabel("Units Sold")
plt.ylabel("Number of days")
plt.tight_layout()
plt.savefig("chart_units_sold_hist.png", dpi=120)
plt.close()


In [12]:
plt.figure(figsize=(9,5))
categories = data["Category"].unique()
box_data = []
for cat in categories:
    box_data.append(data[data["Category"] == cat]["Units Sold"])
plt.boxplot(box_data, labels=categories, showfliers=False)
plt.title("Units Sold by Category")
plt.ylabel("Units Sold")
plt.tight_layout()
plt.savefig("chart_units_sold_by_category.png", dpi=120)
plt.close()

/tmp/ipykernel_578/4044719762.py:6: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(box_data, labels=categories, showfliers=False)


In [13]:
print()
print("Top movers - Top 10 Product IDs by total units sold")
product_totals = data.groupby("Product ID")["Units Sold"].sum()
product_totals = product_totals.sort_values(ascending=False)
print(product_totals.head(10))


Top movers - Top 10 Product IDs by total units sold
Product ID
P0016    508472
P0020    507708
P0014    507622
P0015    507283
P0005    503648
P0009    502086
P0013    500619
P0017    500510
P0011    499362
P0007    499321
Name: Units Sold, dtype: int64


In [14]:
plt.figure(figsize=(8,5))
top10 = product_totals.head(10)
plt.bar(top10.index, top10.values, color="seagreen")
plt.title("Top 10 Products by Total Units Sold")
plt.ylabel("Total Units Sold (2 years)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("chart_top10_products.png", dpi=120)
plt.close()

In [15]:
print()
print("Top movers - Total units sold by Category")
category_totals = data.groupby("Category")["Units Sold"].sum().sort_values(ascending=False)
print(category_totals)


Top movers - Total units sold by Category
Category
Furniture      2025017
Groceries      2000482
Clothing       1999166
Toys           1990485
Electronics    1960432
Name: Units Sold, dtype: int64


In [16]:
print()
print("Dead stock check - lowest 10 Product IDs by total units sold")
print(product_totals.tail(10))


Dead stock check - lowest 10 Product IDs by total units sold
Product ID
P0001    498061
P0019    497899
P0006    497131
P0010    496469
P0004    495501
P0003    493279
P0018    492551
P0012    491670
P0008    488563
P0002    487827
Name: Units Sold, dtype: int64


In [17]:
print()
print("Average Sell_Through_Rate by Product ID (lowest 10)")
sell_through = data.groupby("Product ID")["Sell_Through_Rate"].mean().sort_values()
print(sell_through.head(10))


Average Sell_Through_Rate by Product ID (lowest 10)
Product ID
P0008    0.490583
P0003    0.491187
P0017    0.491940
P0002    0.494488
P0012    0.495053
P0007    0.495844
P0009    0.496803
P0018    0.497185
P0014    0.497206
P0001    0.497820
Name: Sell_Through_Rate, dtype: float64


In [18]:
zero_days = (data["Units Sold"] == 0).sum()
print()
print("Number of store-product-days with 0 units sold:", zero_days)
print("Percent of all rows:", round(zero_days/len(data)*100, 2), "%")


Number of store-product-days with 0 units sold: 360
Percent of all rows: 0.49 %


In [19]:
print()
print("Store-Product combos with most zero-sale days")
zero_data = data[data["Units Sold"] == 0]
zero_counts = zero_data.groupby(["Store ID","Product ID"]).size().sort_values(ascending=False)
print(zero_counts.head(10))


Store-Product combos with most zero-sale days
Store ID  Product ID
S001      P0010         8
S003      P0015         8
S004      P0004         8
S002      P0017         8
          P0018         8
S004      P0007         7
          P0020         7
S003      P0010         7
S001      P0019         6
S003      P0019         6
dtype: int64


In [20]:
print()
print("Correlation of numeric columns with Units Sold")
num_cols = ["Units Sold","Inventory Level","Units Ordered","Demand Forecast","Price",
            "Discount","Competitor Pricing","Sell_Through_Rate","Price_Diff_vs_Competitor","Net_Price"]
corr_table = data[num_cols].corr()
print(corr_table["Units Sold"].sort_values(ascending=False))


Correlation of numeric columns with Units Sold
Units Sold                  1.000000
Demand Forecast             0.996593
Sell_Through_Rate           0.725520
Inventory Level             0.589995
Discount                    0.002576
Competitor Pricing          0.001259
Price                       0.001082
Net_Price                   0.000249
Units Ordered              -0.000930
Price_Diff_vs_Competitor   -0.001674
Name: Units Sold, dtype: float64


In [21]:
plt.figure(figsize=(8,6))
plt.imshow(corr_table, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar()
plt.xticks(range(len(num_cols)), num_cols, rotation=90)
plt.yticks(range(len(num_cols)), num_cols)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig("chart_correlation_heatmap.png", dpi=120)
plt.close()


In [3]:
monthly_avg = data.groupby("Month")["Units Sold"].mean()
print("Average Units Sold by Month")
print(monthly_avg)

plt.figure(figsize=(8,5))
plt.plot(monthly_avg.index, monthly_avg.values, marker="o", color="darkorange")
plt.title("Average Units Sold by Month")
plt.xlabel("Month")
plt.ylabel("Average Units Sold")
plt.xticks(range(1,13))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("chart_monthly_seasonality.png", dpi=120)
plt.close()

Average Units Sold by Month
Month
1     135.966667
2     138.610714
3     135.902419
4     134.742833
5     134.430968
6     136.858333
7     139.443065
8     135.712419
9     136.161833
10    137.501452
11    138.443500
12    134.031129
Name: Units Sold, dtype: float64


In [4]:
weekday_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
weekday_avg = data.groupby("Weekday")["Units Sold"].mean()
weekday_avg = weekday_avg.reindex(weekday_order)
print()
print("Average Units Sold by Weekday")
print(weekday_avg)

plt.figure(figsize=(8,5))
plt.bar(weekday_avg.index, weekday_avg.values, color="cornflowerblue")
plt.title("Average Units Sold by Weekday")
plt.ylabel("Average Units Sold")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig("chart_weekday_seasonality.png", dpi=120)
plt.close()


Average Units Sold by Weekday
Weekday
Monday       135.094667
Tuesday      137.307692
Wednesday    136.615096
Thursday     137.281923
Friday       136.851442
Saturday     135.314476
Sunday       136.809714
Name: Units Sold, dtype: float64


In [5]:
season_avg = data.groupby("Seasonality")["Units Sold"].mean().sort_values(ascending=False)
print()
print("Average Units Sold by Season label in data")
print(season_avg)


Average Units Sold by Season label in data
Seasonality
Autumn    137.782444
Winter    136.830790
Spring    135.826828
Summer    135.428298
Name: Units Sold, dtype: float64


In [6]:
promo_avg = data.groupby("Holiday/Promotion")["Units Sold"].mean()
print()
print("Average Units Sold: 0 = normal day, 1 = holiday/promotion day")
print(promo_avg)

plt.figure(figsize=(5,5))
plt.bar(["Normal day","Holiday/Promo day"], promo_avg.values, color=["gray","tomato"])
plt.title("Effect of Holiday/Promotion on Units Sold")
plt.ylabel("Average Units Sold")
plt.tight_layout()
plt.savefig("chart_promo_effect.png", dpi=120)
plt.close()


Average Units Sold: 0 = normal day, 1 = holiday/promotion day
Holiday/Promotion
0    136.505375
1    136.423926
Name: Units Sold, dtype: float64


In [7]:
pivot = data.pivot_table(index="Category", columns="Month", values="Units Sold", aggfunc="mean")
print()
print("Category x Month average units sold table")
print(pivot)

plt.figure(figsize=(9,5))
plt.imshow(pivot, cmap="YlGnBu", aspect="auto")
plt.colorbar(label="Average Units Sold")
plt.yticks(range(len(pivot.index)), pivot.index)
plt.xticks(range(len(pivot.columns)), pivot.columns)
plt.xlabel("Month")
plt.title("Category vs Month - Average Units Sold")
plt.tight_layout()
plt.savefig("chart_category_month_heatmap.png", dpi=120)
plt.close()


Category x Month average units sold table
Month                1           2           3           4           5   \
Category                                                                  
Clothing     134.032122  139.837542  130.360163  136.830372  132.759703   
Electronics  137.906225  130.322129  135.892979  129.747432  136.357318   
Furniture    137.681310  137.110631  139.452140  137.761246  131.521992   
Groceries    136.769823  141.095325  139.135135  133.377998  135.673653   
Toys         133.373940  144.506862  134.331424  135.890968  135.666122   

Month                6           7           8           9           10  \
Category                                                                  
Clothing     138.756778  143.900566  136.487550  136.663391  134.078767   
Electronics  136.026293  135.227492  135.958202  134.921585  140.809865   
Furniture    137.864979  139.219473  135.141720  132.908638  141.807390   
Groceries    138.466777  142.861603  133.801533  141.391

In [8]:
weather_avg = data.groupby("Weather Condition")["Units Sold"].mean().sort_values(ascending=False)
print()
print("Average Units Sold by Weather Condition")
print(weather_avg)


Average Units Sold by Weather Condition
Weather Condition
Sunny     138.028650
Cloudy    136.758324
Snowy     135.911559
Rainy     135.160028
Name: Units Sold, dtype: float64


In [22]:
data = data.sort_values(["Store ID", "Product ID", "Date"])
data = data.reset_index(drop=True)

In [23]:
group_cols = ["Store ID", "Product ID"]
grouped = data.groupby(group_cols)["Units Sold"]

In [24]:
data["lag_1"] = grouped.shift(1)
data["lag_7"] = grouped.shift(7)
data["lag_14"] = grouped.shift(14)
data["lag_28"] = grouped.shift(28)

In [25]:
shifted_sales = grouped.shift(1)  # shift first so today's value is not included in its own rolling window
data["rolling_mean_7"] = shifted_sales.groupby([data["Store ID"], data["Product ID"]]).transform(lambda x: x.rolling(7).mean())
data["rolling_std_7"] = shifted_sales.groupby([data["Store ID"], data["Product ID"]]).transform(lambda x: x.rolling(7).std())
data["rolling_mean_28"] = shifted_sales.groupby([data["Store ID"], data["Product ID"]]).transform(lambda x: x.rolling(28).mean())

In [26]:
data["day_of_week"] = data["Date"].dt.dayofweek   # Monday=0 ... Sunday=6
data["is_weekend"] = (data["day_of_week"] >= 5).astype(int)
data["week_of_year"] = data["Date"].dt.isocalendar().week.astype(int)
data["month"] = data["Date"].dt.month
data["quarter"] = data["Date"].dt.quarter

In [27]:
data["promo_flag"] = data["Holiday/Promotion"]
# promo yesterday (helps model catch after-promo dip or bump)
data["promo_lag_1"] = data.groupby(group_cols)["Holiday/Promotion"].shift(1)
# rolling count of promo days in the last 7 days
data["promo_count_7"] = data.groupby(group_cols)["Holiday/Promotion"].transform(lambda x: x.shift(1).rolling(7).sum())

In [28]:
print("New columns added:")
new_cols = ["lag_1","lag_7","lag_14","lag_28","rolling_mean_7","rolling_std_7",
            "rolling_mean_28","day_of_week","is_weekend","week_of_year","month",
            "quarter","promo_flag","promo_lag_1","promo_count_7"]
print(new_cols)

New columns added:
['lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_std_7', 'rolling_mean_28', 'day_of_week', 'is_weekend', 'week_of_year', 'month', 'quarter', 'promo_flag', 'promo_lag_1', 'promo_count_7']


In [29]:
print()
print("Example rows for one store-product (first 10 rows will have some empty lag values, that is normal):")
example = data[(data["Store ID"]=="S001") & (data["Product ID"]=="P0001")]
print(example[["Date","Units Sold"] + new_cols].head(10))


Example rows for one store-product (first 10 rows will have some empty lag values, that is normal):
        Date  Units Sold  lag_1  lag_7  lag_14  lag_28  rolling_mean_7  \
0 2022-01-01         127    NaN    NaN     NaN     NaN             NaN   
1 2022-01-02          81  127.0    NaN     NaN     NaN             NaN   
2 2022-01-03           5   81.0    NaN     NaN     NaN             NaN   
3 2022-01-04          58    5.0    NaN     NaN     NaN             NaN   
4 2022-01-05         147   58.0    NaN     NaN     NaN             NaN   
5 2022-01-06          37  147.0    NaN     NaN     NaN             NaN   
6 2022-01-07         107   37.0    NaN     NaN     NaN             NaN   
7 2022-01-08           2  107.0  127.0     NaN     NaN       80.285714   
8 2022-01-09         350    2.0   81.0     NaN     NaN       62.428571   
9 2022-01-10          36  350.0    5.0     NaN     NaN      100.857143   

   rolling_std_7  rolling_mean_28  day_of_week  is_weekend  week_of_year  \
0       

In [30]:
data.to_csv("features_data.csv", index=False)
print()
print("Saved features_data.csv with shape:", data.shape)


Saved features_data.csv with shape: (73100, 39)


In [31]:
data.to_csv("features_data.csv", index=False)
print()
print("Saved features_data.csv with shape:", data.shape)


Saved features_data.csv with shape: (73100, 39)


In [32]:
data = data.sort_values(["Store ID", "Product ID", "Date"])
data = data.reset_index(drop=True)

group_cols = ["Store ID", "Product ID"]

In [33]:
season_length = 7
data["seasonal_naive_forecast"] = data.groupby(group_cols)["Units Sold"].shift(season_length)



In [34]:
scoring_data = data.dropna(subset=["seasonal_naive_forecast"]).copy()

print("Rows available for scoring (after removing first week of each series):", len(scoring_data))


Rows available for scoring (after removing first week of each series): 72400


In [35]:
def wape(actual, forecast):
    actual = actual.values
    forecast = forecast.values
    error = abs(actual - forecast).sum()
    total_actual = abs(actual).sum()
    return error / total_actual * 100

overall_wape = wape(scoring_data["Units Sold"], scoring_data["seasonal_naive_forecast"])
print()
print("Overall seasonal-naive baseline WAPE: {:.2f}%".format(overall_wape))


Overall seasonal-naive baseline WAPE: 87.84%


In [36]:
print()
print("WAPE by Category:")
for cat in scoring_data["Category"].unique():
    subset = scoring_data[scoring_data["Category"] == cat]
    cat_wape = wape(subset["Units Sold"], subset["seasonal_naive_forecast"])
    print(cat, "-> {:.2f}%".format(cat_wape))



WAPE by Category:
Furniture -> 88.16%
Electronics -> 87.57%
Toys -> 88.00%
Groceries -> 88.41%
Clothing -> 87.08%


In [37]:
print()
print("WAPE by Store:")
for store in sorted(scoring_data["Store ID"].unique()):
    subset = scoring_data[scoring_data["Store ID"] == store]
    store_wape = wape(subset["Units Sold"], subset["seasonal_naive_forecast"])
    print(store, "-> {:.2f}%".format(store_wape))



WAPE by Store:
S001 -> 88.53%
S002 -> 88.06%
S003 -> 87.52%
S004 -> 87.54%
S005 -> 87.56%


In [38]:
data["naive_lag1_forecast"] = data.groupby(group_cols)["Units Sold"].shift(1)
compare_data = data.dropna(subset=["seasonal_naive_forecast", "naive_lag1_forecast"]).copy()

lag1_wape = wape(compare_data["Units Sold"], compare_data["naive_lag1_forecast"])
seasonal_wape_compare = wape(compare_data["Units Sold"], compare_data["seasonal_naive_forecast"])


In [39]:
print()
print("Baseline comparison on the same rows:")
print("Naive (yesterday's value) WAPE: {:.2f}%".format(lag1_wape))
print("Seasonal-naive (7 days ago) WAPE: {:.2f}%".format(seasonal_wape_compare))


Baseline comparison on the same rows:
Naive (yesterday's value) WAPE: 87.94%
Seasonal-naive (7 days ago) WAPE: 87.84%


In [40]:
scoring_data.to_csv("baseline_predictions.csv", index=False)
print()
print("Saved baseline_predictions.csv")



Saved baseline_predictions.csv
